# RAG de receitas, passo a passo

Demonstração da versão 1 do RAG do Miau-ricio (épico #15, entrega da semana 2): a consulta enviada, os trechos recuperados, a resposta baseada neles e as fontes usadas.

A estratégia completa está em [docs/semana-2/estrategia-rag.md](../docs/semana-2/estrategia-rag.md). Antes de rodar:

1. `docker compose up -d` (Qdrant);
2. Ollama aberto, com `qwen2.5:7b` e `qwen3-embedding:4b`;
3. dados em `data/raw/` (comandos na seção "Dados" do [plano](../specs/15-rag-receitas/plano-v1.md));
4. base indexada: `poetry run python -m economia.receitas.cli indexar --modelo e5`.

In [1]:
import os
from pathlib import Path

# o código lê os dados a partir da raiz do repositório
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd().name)

challenge2-qualidade-de-vida


## 1. Documentos-fonte e curadoria

As receitas vêm do afrodite.json (português) e de uma amostra fixa do RecipeNLG (inglês). Saem as receitas sem ingredientes ou sem modo de preparo e as duplicatas. A orientação vem do Guia Alimentar e de quatro textos do Ministério da Saúde e da Embrapa (ver `data/base-conhecimento/FONTES.md`).

In [2]:
from economia.receitas.cli import (
    ARQUIVO_AFRODITE,
    ARQUIVO_RECIPENLG,
    carregar_documentos,
)
from economia.receitas.infraestrutura.carregadores import (
    carregar_afrodite,
    carregar_recipenlg,
)

afrodite = carregar_afrodite(ARQUIVO_AFRODITE)
recipenlg = carregar_recipenlg(ARQUIVO_RECIPENLG, quantidade=3000)
documentos = carregar_documentos()

print(
    f"afrodite.json: {len(afrodite)} receitas sem duplicatas, {sum(r.esta_completa for r in afrodite)} completas"
)
print(f"RecipeNLG: {len(recipenlg)} receitas na amostra")
for documento in documentos:
    print(f"{documento.titulo}: {len(documento.secoes)} seções")

afrodite.json: 7179 receitas sem duplicatas, 1004 completas
RecipeNLG: 3000 receitas na amostra
Guia Alimentar para a População Brasileira: 76 seções
Na cozinha com as frutas, legumes e verduras: 108 seções
Hortaliça não é só salada: Geladeira: 2 seções
Hortaliça não é só salada: Congelador: 2 seções
Recomendações para compras, armazenamento e consumo de frutas e hortaliças: 21 seções


## 2. Chunking

- **Receita = 1 trecho:** título e ingredientes primeiro, cortado em 512 tokens (limite do e5). O texto completo fica no payload.
- **Orientação:** por seção e por parágrafo, até cerca de 350 tokens, com 50 de sobreposição. Um trecho nunca junta duas seções; cada um dos Dez passos do Guia é um trecho próprio.

Os tokens são contados com o tokenizador do e5, e os dois modelos recebem os mesmos trechos.

In [3]:
from economia.receitas.aplicacao.fragmentacao import (
    fragmentar_documento,
    fragmentar_receita,
)
from economia.receitas.infraestrutura.embeddings_e5 import contador_de_tokens_e5

contar = contador_de_tokens_e5()
receita = next(r for r in afrodite if r.esta_completa)
trecho = fragmentar_receita(receita, contar=contar)
print(trecho.texto[:600])
print("\ntokens:", contar(trecho.texto))

Receita: Brownie de Chocolate com Gengibre. Ingredientes: 50 g farinha de milho fina; 10 g de cacau em pó; 250 g de chocolate meio amargo; 200 g de manteiga sem sal cortada em cubos; 20 ml de suco de gengibre; 5 ovos; 200 g de açúcar; 1 colher (chá) de fermento em pó; 100 g de nozes picadas grosseiramente. Modo de preparo: Coloque numa tigela a farinha de milho fina e o cacau em pó. Misture e reserve. Numa panela, em banho-maria, derreta o chocolate meio amargo picado com a manteiga sem sal cortada em cubos. Retire do fogo. Adicione o suco de gengibre e misture. Acrescente a mistura de farinha

tokens: 403


In [4]:
guia = documentos[0]
trechos_do_guia = fragmentar_documento(guia, contar=contar)
passo = next(
    t for t in trechos_do_guia if (t.secao or "").startswith("Evitar o consumo")
)
print(f"{len(trechos_do_guia)} trechos do Guia")
print(passo.capitulo, "|", passo.secao, "| p.", passo.pagina)
print(passo.texto)

167 trechos do Guia
Dez passos para uma alimentação adequada e saudável | Evitar o consumo de alimentos ultraprocessados | p. 126
Devido a seus ingredientes, alimentos ultraprocessados – como biscoitos recheados, “salgadinhos de pacote”, refrigerantes e “macarrão instantâneo” – são nutricionalmente desbalanceados. Por conta de sua formulação e apresentação, tendem a ser consumidos em excesso e a substituir alimentos in natura ou minimamente processados. suas formas de produção, distribuição, comercialização e consumo afetam de modo desfavorável a cultura, a vida social e o meio ambiente.


## 3. Embeddings e banco vetorial

O texto vira um vetor de 768 dimensões (e5) ou 2.560 (qwen3), normalizado. O Qdrant guarda o vetor e o payload do trecho, uma coleção por modelo, e busca pela similaridade de cosseno.

In [5]:
from economia.compartilhado.configuracao import obter_configuracao
from economia.receitas.cli import criar_gerador, criar_repositorio

configuracao = obter_configuracao()
gerador = criar_gerador(configuracao)
repositorio = criar_repositorio(configuracao)
print(configuracao.colecao, "→", repositorio.contar(), "trechos")

vetor = gerador.gerar_para_pergunta("como conservo alface para durar mais?")
print(len(vetor), "dimensões; primeiras:", [round(x, 3) for x in vetor[:5]])

base_conhecimento_e5 → 4346 trechos


768 dimensões; primeiras: [0.007, 0.037, -0.006, 0.046, 0.032]


## 4. Consulta e trechos recuperados

A busca devolve os 5 trechos mais parecidos. Abaixo do limiar de similaridade (RN11), a resposta diz que não encontrou.

In [6]:
from economia.receitas.cli import criar_responder

caso = criar_responder(configuracao)
pergunta = "o que posso fazer com ovo, cenoura e farinha?"
for n, r in enumerate(
    repositorio.buscar(gerador.gerar_para_pergunta(pergunta), 5), start=1
):
    acima = "acima" if r.score >= configuracao.limiar_similaridade else "abaixo"
    print(
        f"{n}. {r.score:.3f} ({acima} do limiar) {r.trecho.tipo.value}: {r.trecho.titulo}"
    )

1. 0.874 (acima do limiar) receita: Bolo cremoso de cenouras
2. 0.861 (acima do limiar) receita: Bolo Prático de Cenoura
3. 0.860 (acima do limiar) conhecimento: Na cozinha com as frutas, legumes e verduras
4. 0.859 (acima do limiar) receita: Massa Salgada de Arroz
5. 0.858 (acima do limiar) receita: Bolo de Cenoura de Micro-ondas


## 5. Resposta com as fontes

O LLM (`qwen2.5:7b`, local) recebe os trechos numerados e cita os números. O código troca os números pelas fontes e acrescenta os ingredientes que faltam em cada receita, contados por função determinística (RN01, RN10, RN31 proposta). Na CPU, a resposta leva de 1 a 3 minutos.

In [7]:
from economia.receitas.aplicacao.responder_pergunta import formatar_fonte

resposta = caso.responder(pergunta)
print(resposta.texto)
print("\nFontes:")
for n, r in enumerate(resposta.trechos, start=1):
    print(f"[{n}] {formatar_fonte(r)}")

Bolo Cremoso de Cenouras [1]

Modo de preparo:
Limpe bem e corte em pedaços meio quilo de cenouras. Ferva até amaciar. Depois de frias, bata no liquidificador quatro gemas e um ovo inteiro, com uma parte de 400 grs de açúcar e vá juntando aos poucos a cenoura, com o aparelho em movimento. À parte, bata as quatro claras em neve. Depois que tiver batido as gemas com as cenouras, junte numa bacia onde já tenha colocado o restante dos 400 grs do açúcar, cinco colheres (de sopa, bem cheias) de farinha de trigo, 100 grs de coco ralado seco, duas colheres (de chá) de fermento em pó, e por último, as claras em neve. Mexa delicadamente. Vai para forma, untada e polvilhada com farinha, em forno médio (180 graus), por mais ou menos 45 minutos. Quando estiver assado, retire do forno e polvilhe com açúcar e bastante canela, generosamente.

Ingredientes que faltam:
- Bolo cremoso de cenouras [1]: 4 gemas, 400 grs de açúcar, 4 claras, 100 grs de coco ralado seco, 2 colheres (de chá) de fermento em pó

## 6. Sem informação na base e preço atual

Sem trecho acima do limiar, o sistema diz que não encontrou (RN11). Pergunta sobre preço atual não passa pela base (RN04).

In [8]:
for pergunta in (
    "quanto tempo dura o fermento biológico depois de aberto?",
    "quanto está o arroz no mercado hoje?",
):
    print(pergunta, "→", caso.responder(pergunta).texto, "\n")

quanto tempo dura o fermento biológico depois de aberto? → Não encontrei essa informação na base de conhecimento. Prefiro não responder a inventar uma resposta. 

quanto está o arroz no mercado hoje? → Preços atuais não vêm da base de conhecimento de receitas e alimentação. Para saber quanto um produto custa hoje, consulte os preços cadastrados dos mercados. 



## 7. Comparação dos modelos de embeddings

Resultado do comando `comparar` no conjunto `data/avaliacao/perguntas_rag_v1.json`: Recall@5, MRR, separação dos scores entre perguntas com e sem resposta e latência por consulta.

In [9]:
import json

for arquivo in sorted(Path("data/avaliacao").glob("resultados_v1*.json")):
    dados = json.loads(arquivo.read_text())
    print(f"== {arquivo.name} ({dados['maquina']})")
    for m in dados["modelos"]:
        print(
            f"{m['colecao']:<38} Recall@5 {m['recall']:.2f}  MRR {m['mrr']:.2f}  "
            f"separação {m['separacao']:+.3f}  limiar sugerido {m['limiar_sugerido']}  "
            f"{m['latencia_media']:.2f} s/consulta"
        )

== resultados_v1.json (Linux x86_64, 8 CPUs lógicas)
base_conhecimento_e5                   Recall@5 0.76  MRR 0.63  separação -0.005  limiar sugerido 0.842  0.46 s/consulta
